# M3 CLV 조건부 후보상품 관계 실패 진단 — Dunnhumby seed 42

이미 학습된 일반 후보관계·실제 historical CLV·degree-matched CLV shuffle checkpoint를 재사용합니다. 재학습이나 checkpoint 선택 없이 실패를 **후보 생성 → 보조점수 전달 → Top-K 진입**으로 분해합니다.

- 학습: 없음
- 분석 구간: `DAY 1~683` 학습 결과와 `DAY 684~690` 개발평가 정답
- final test·holdout: 생성하지 않음
- 용도: 다음 M3에서 수정할 한 지점을 고르는 사후 기술진단이며 유의성·인과·일반화를 주장하지 않음

## 1. 고정 소스와 Drive 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = 'e2ca7f42ce6554f94692889d63a6538f2ed3096d'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run([
    'git', 'clone', '-q',
    'https://github.com/jung-un/clv-m2-lightgcn-runner.git',
    str(repo),
], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
for name in list(sys.modules):
    if name.startswith(('lightgcn_clv', 'clv_m3', 'clv_run_state')):
        sys.modules.pop(name, None)
importlib.invalidate_caches()
print('Pinned execution source:', actual_sha)

## 2. 분석 계약 확인

In [ ]:
import inspect, json, torch
import lightgcn_clv_m3_candidate_item_diagnostic as diagnostic

diagnostic = importlib.reload(diagnostic)
assert diagnostic.CODE_VERSION == 'm3-clv-candidate-item-failure-diagnostic-v1'
assert str(Path(diagnostic.__file__).resolve()).startswith(str(repo.resolve()))
loaded_source = inspect.getsource(diagnostic.run_m3_candidate_item_diagnostic)
assert 'candidate_truth_coverage' in loaded_source
assert 'diagnostic_route' in loaded_source

cfg = diagnostic.configure_m3_candidate_item_diagnostic()
summary = diagnostic.preflight_summary(cfg)
assert summary['training'] is False
assert summary['checkpoint_selection'] is False
assert summary['final_test_constructed'] is False
assert summary['holdout_constructed'] is False
assert summary['source_result_id'] == 'd5c0423bfd90'
assert summary['rank_limit'] == 50
assert not torch.is_grad_enabled() or summary['training'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 3. Checkpoint 기반 진단 실행

그래프를 다시 구성하고 세 checkpoint에서 Top-50과 점수 구성요소를 복원하므로 몇 분이 걸릴 수 있습니다. 모델 학습은 하지 않습니다.

In [ ]:
graph_summary = diagnostic.run_m3_candidate_item_diagnostic(cfg)
RESULT_PATHS = graph_summary.attrs['result_paths']
print(json.dumps(RESULT_PATHS, ensure_ascii=False, indent=2))

## 4. 입력·checkpoint·추천목록 품질검사

In [ ]:
import pandas as pd
from IPython.display import display

tables = {
    name: pd.read_csv(path)
    for name, path in RESULT_PATHS.items()
    if path.endswith('.csv')
}
display(tables['quality_checks_csv'])
assert tables['quality_checks_csv']['passed'].all(), '품질검사가 실패했습니다.'

## 5. 1단계 — 실제 CLV 후보 100개에 정답이 더 많이 들어오는가

`candidate_truth_pair_coverage`는 전체 정답쌍 중 보조후보에 포함된 비율이고, `macro_candidate_truth_recall`은 사용자별 정답 포함률의 평균입니다. 실제 CLV가 shuffle보다 높지 않으면 다음 모델은 **CLV→상품 후보관계 정의**만 수정해야 하며 `gamma`나 BPR을 바꾸면 안 됩니다.

In [ ]:
candidate_summary = tables['candidate_truth_summary_csv']
display(candidate_summary.sort_values(['clv_group', 'graph_arm']))
display(tables['candidate_truth_comparison_csv'])

plot_data = candidate_summary.pivot(
    index='clv_group', columns='graph_arm', values='candidate_truth_pair_coverage'
).reindex(['전체', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
ax = plot_data.plot(kind='bar', figsize=(10, 4))
ax.set_title('Held-out truth coverage inside 100 auxiliary candidates')
ax.set_xlabel('Train-only historical CLV quintile')
ax.set_ylabel('Truth-pair coverage')
ax.set_ylim(bottom=0)
ax.figure.tight_layout()

## 6. 실제 CLV와 shuffle 후보행은 실제로 다른가

Jaccard가 낮고 총변동거리가 크면 후보 그래프는 달랐다는 뜻입니다. 후보행이 다른데 정답 포함률이 같다면 개입 부재가 아니라 **관계 방향의 무용성**입니다.

In [ ]:
display(tables['graph_similarity_summary_csv'])
display(tables['clv_assignment_correlation_csv'])

## 7. 2단계 — 후보관계가 정답 점수로 전달되는가

`truth_minus_top50_auxiliary_mean`이 클수록 보조채널이 경쟁 Top-50 상품보다 실제 정답에 더 큰 점수를 줍니다. 후보 정답 포함률은 actual이 앞서지만 이 값이 shuffle보다 낮다면 **후보행은 유지하고 메시지 중심화·전달 방식만** 수정합니다.

In [ ]:
score_contrast = tables['score_truth_candidate_contrast_csv']
display(score_contrast.sort_values(['clv_group', 'model_id']))

plot_data = score_contrast.pivot(
    index='clv_group', columns='model_id', values='truth_minus_top50_auxiliary_mean'
).reindex(['전체', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
ax = plot_data.plot(kind='bar', figsize=(11, 4))
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Auxiliary score: held-out truths minus competitive Top-50 items')
ax.set_xlabel('Train-only historical CLV quintile')
ax.set_ylabel('Mean auxiliary-score contrast')
ax.figure.tight_layout()

## 8. 3단계 — 점수 차이가 Top-K 추천을 바꾸는가

후보와 점수 방향은 actual이 앞서지만 추천집합 변경률이 0이면 메시지 정규화 또는 결합 강도만 수정합니다. 추천집합은 바뀌는데 CLV 귀속 성과가 실패했다면 점수 전달은 되고 있으므로 강도를 키우지 말고 **어떤 정답이 들어오고 빠졌는지**를 점검해야 합니다.

In [ ]:
recommendations = tables['recommendation_overlap_csv']
display(recommendations.sort_values(['comparison', 'clv_group', 'k']))

focus = recommendations[
    (recommendations['comparison'] == 'actual_full_vs_shuffle_full')
    & (recommendations['clv_group'] == '전체')
].set_index('k')
ax = focus[['set_changed_user_share', 'order_changed_user_share']].plot(
    kind='bar', figsize=(8, 4)
)
ax.set_title('Actual CLV vs shuffle recommendation changes')
ax.set_xlabel('Rank cutoff')
ax.set_ylabel('Evaluation-user share')
ax.set_ylim(bottom=0)
ax.figure.tight_layout()

## 9. 결과에 따라 다음 M3에서 수정할 한 지점

In [ ]:
reading = graph_summary.attrs['diagnostic_reading']
print(json.dumps(reading, ensure_ascii=False, indent=2))

NEXT_CHANGE_KO = {
    'candidate_relation_construction': (
        '후보 생성식 수정: CLV→상품 관계 통계와 지지도 집계만 바꾸고, '
        'gamma·optimizer·BPR은 고정합니다. 현재 후보행을 단순 증폭하지 않습니다.'
    ),
    'relation_to_score_transfer': (
        '점수 전달 수정: 후보상품 집합은 유지하고 pooled 일반관계 대비 메시지를 '
        '중심화한 뒤 같은 forward graph에서 전달하는 방식만 바꿉니다.'
    ),
    'score_to_rank_boundary': (
        'Top-K 경계 수정: 그래프 방향은 유지하고 새 사전고정 실행에서만 제한된 '
        '정규화 또는 결합 강도를 변경합니다.'
    ),
    'ranking_alignment': (
        '순위 방향 수정: CLV 채널은 Top-K를 바꾸고 있으므로 gamma를 키우지 말고, '
        'CLV 분위별로 들어온 정답과 빠진 정답을 먼저 확인한 뒤 관계 의미를 고칩니다.'
    ),
}
print('\n다음 수정 위치:')
print(NEXT_CHANGE_KO[reading['descriptive_bottleneck']])
print('\n주의: 같은 개발구간에서 수치를 맞추지 않고, 변경 모델은 새 사전고정 구간 또는 독립 데이터로 평가합니다.')